# KOVA Voice Studio - Google Colab GPU Worker

Select **Runtime > Change runtime type > GPU**, then use **Runtime > Run all**. This worker refuses CPU fallback. The final cell prints a temporary HTTPS URL and bearer token to paste into KOVA Desktop.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/khoinguyen59/kova-video-dubbing.git'
REPO_REF = 'main'  # Replace with a KOVA release tag for production sessions.
WORKSPACE = Path('/content/kova-video-dubbing')

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

run(['nvidia-smi'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.8.13'])
if not WORKSPACE.exists():
    run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(WORKSPACE)])
VOICE_DIR = WORKSPACE / 'voice-studio'
if not VOICE_DIR.exists():
    raise RuntimeError(f'Voice Studio source is missing: {VOICE_DIR}')
run(['uv', 'python', 'install', '3.11'])
run(['uv', 'venv', '--python', '3.11', '.venv'], cwd=VOICE_DIR)
PYTHON = str(VOICE_DIR / '.venv' / 'bin' / 'python')
OMNIVOICE_DIR = Path('/content/OmniVoice')
if not OMNIVOICE_DIR.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/k2-fsa/OmniVoice.git', str(OMNIVOICE_DIR)])
run(['uv', 'pip', 'install', '--python', PYTHON, '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.8.0+cu128', 'torchaudio==2.8.0+cu128'])
run(['uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements-colab.txt', 'demucs==4.0.1'], cwd=VOICE_DIR)
run(['uv', 'pip', 'install', '--python', PYTHON, str(OMNIVOICE_DIR)])
run(['uv', 'pip', 'install', '--python', PYTHON, '-e', '.'], cwd=VOICE_DIR)

## KOVA API Gateway (translation and preset TTS)

KOVA Desktop uses the gateway for free translation models and Google/Edge preset TTS. Voice Studio itself remains the separate OmniVoice cloning worker, so this notebook never copies a gateway key into the repository or a voice profile. If a Colab process later starts KOVA itself, store the key in the Colab secret named `KOVA_API_GATEWAY_API_KEY`; the code reads it only from runtime memory.

In [ ]:
GATEWAY_BASE_URL = 'http://3.27.172.90/v1'
FREE_TRANSLATION_MODELS = (
    'oc/deepseek-v4-flash-free', 'oc/mimo-v2.5-free', 'oc/hy3-free',
    'oc/big-pickle', 'oc/nemotron-3-ultra-free', 'oc/north-mini-code-free',
)
try:
    from google.colab import userdata
    gateway_key = userdata.get('KOVA_API_GATEWAY_API_KEY')
except Exception:
    gateway_key = None
if gateway_key:
    os.environ['KOVA_API_GATEWAY_API_KEY'] = gateway_key
    print('KOVA API Gateway secret loaded into this Colab runtime only.')
else:
    print('Gateway secret not loaded. This is fine for the Voice Studio worker; add the Colab secret only when KOVA runs in this runtime.')
print('Gateway base URL:', GATEWAY_BASE_URL)
print('Free translation models:', ', '.join(FREE_TRANSLATION_MODELS))
print('Verified preset TTS models: google-tts/vi, google-tts/en, edge-tts/vi-VN-HoaiMyNeural, edge-tts/vi-VN-NamMinhNeural')

In [ ]:
# KOVA Voice Studio notebook revision: 2026-07-22.
# Fail before model download if Colab resolves an incompatible package.
# IMPORTANT: this is Python __version__ (two underscores), never Markdown **version**.
import json
CHECK = r'''
import json, torch, transformers
from transformers import HiggsAudioV2TokenizerModel
from omnivoice import OmniVoice
assert torch.cuda.is_available(), 'CUDA is unavailable; stop and select a GPU runtime.'
version = getattr(transformers, '__version__', '')
assert version == '5.3.0', version
print(json.dumps({'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'transformers': version}))
'''
doctor = subprocess.run([PYTHON, '-c', CHECK], cwd=VOICE_DIR, text=True, capture_output=True)
print(doctor.stdout, end='')
if doctor.returncode:
    raise RuntimeError('Dependency doctor failed. Reopen the current KOVA notebook from GitHub, then Run all.\n--- stderr ---\n' + doctor.stderr[-6000:])
print('Dependency doctor passed. The worker loads the model only on the first synthesis request.')

In [ ]:
import secrets, time, urllib.request, urllib.parse

TOKEN = secrets.token_urlsafe(32)
PAIR_CODE = secrets.token_urlsafe(32)
ENV = os.environ.copy()
ENV.update({
    'KOVA_VOICE_API_TOKEN': TOKEN,
    'KOVA_VOICE_PAIR_CODE': PAIR_CODE,
    'KOVA_VOICE_REQUIRE_CUDA': '1',
    'KOVA_VOICE_SEPARATE_REFERENCE': '1',
    'KOVA_VOICE_DATA_DIR': '/content/kova-voice-data',
})
PAIR_WORKER = VOICE_DIR / 'kova_pair_worker.py'
PAIR_WORKER.write_text('''import os, secrets
from threading import Lock
from fastapi import HTTPException
from kova_voice_studio.api import create_app

app = create_app()
_pair_lock = Lock()
_pair_claimed = False

@app.get("/v1/pairing/{code}")
def claim_pairing(code: str):
    global _pair_claimed
    expected = os.environ.get("KOVA_VOICE_PAIR_CODE", "")
    token = os.environ.get("KOVA_VOICE_API_TOKEN", "")
    if not expected or not token or not secrets.compare_digest(code, expected):
        raise HTTPException(status_code=404, detail="pairing link is invalid or expired")
    with _pair_lock:
        if _pair_claimed:
            raise HTTPException(status_code=410, detail="pairing link was already used")
        _pair_claimed = True
    return {"token": token}
''', encoding='utf-8')
LOG = '/content/kova-voice-worker.log'
worker = subprocess.Popen([PYTHON, '-m', 'uvicorn', 'kova_pair_worker:app', '--host', '127.0.0.1', '--port', '3920'], cwd=VOICE_DIR, env=ENV, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        request = urllib.request.Request('http://127.0.0.1:3920/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            health = json.load(response)
        if health.get('ready') and health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Voice worker did not become CUDA-ready. Log tail:\n' + Path(LOG).read_text(errors='replace')[-4000:])

run(['wget', '-q', '-O', '/content/cloudflared.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared.deb'])
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3920', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    marker = 'https://'
    if marker in line and 'trycloudflare.com' in line:
        public_url = line[line.find(marker):].split()[0]
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found.')
print('\nKOVA_VOICE_URL=' + public_url)
pairing_link = 'kova-voice-studio://pair?worker_url=' + urllib.parse.quote(public_url, safe='') + '&code=' + urllib.parse.quote(PAIR_CODE, safe='')
from IPython.display import HTML, Javascript, display
display(HTML('<div style="font-family:Arial,sans-serif;padding:12px 16px;border-radius:8px;background:#eef6ff;color:#123;"><strong>KOVA đang tự kết nối với desktop...</strong><br><a id="kova-pairing-link" href="' + pairing_link + '" style="display:inline-block;margin-top:8px;padding:10px 14px;border-radius:7px;background:#2f80ed;color:#fff;font-weight:700;text-decoration:none">Mở kết nối thủ công nếu cần</a></div>'))
display(Javascript("const kovaPairingURL = " + repr(pairing_link) + "; setTimeout(() => { window.open(kovaPairingURL, '_blank'); }, 300);"))
print('DEVICE=cuda')
print('KOVA tự gửi URL và mã ghép nối dùng một lần về desktop sau khi cell này chạy; không cần sao chép URL hoặc token.')